<!-- notebook-header -->
<a id="top"></a>

# 9. Graphs

*LeetCode Playbook · notebook 9 of 12* · BFS and DFS, ordering and connectivity, weighted shortest paths and spanning trees.

**In this notebook:** [Graphs I: BFS & DFS](#s17) · [Graphs II: Ordering & Connectivity](#s18) · [Graphs III: Weighted Paths & MST](#s19)

← [Backtracking (and a short DP map)](08_Backtracking.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Matrices, Math & Bits](10_Matrices_Math_Bits.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s17"></a>

## Graphs I: BFS & DFS

> A graph is just *things* (nodes) and *you can get from this one to that one* (edges). BFS spreads from the start like a ripple in a pond, one ring of equal distance at a time; DFS runs down one corridor to its end before backing up. Both visit every node they can reach exactly once, because they mark a node the moment they discover it.

**Reach for it when** the problem talks about cells connected up/down/left/right, islands or regions, something spreading in rounds (rot, fire, a signal), "fewest steps / moves / jumps / transformations" where every move costs the same, "can I reach", copying a structure that has cycles, or a situation that changes one move at a time (a board, a word, a key ring).

**In this repo:** `graphs/` (19 of its 45 problems; Graphs II and III take the rest) · bank: `practice/simple/41_number_of_islands.py`, `practice/simple/42_clone_graph.py`, `practice/simple/44_rotting_oranges.py` · basics: `practice/simple/basics/graphs/01_adjacency_list_bfs_dfs.py`, `practice/simple/basics/searches/03_bfs_grid_shortest_path.py`, `practice/simple/basics/searches/04_dfs_recursive_and_iterative.py`, `practice/simple/basics/searches/05_connected_components.py`, `practice/simple/basics/searches/06_multi_source_bfs_01_matrix.py`

### The picture

Half of every graph problem is one sentence: *what is a node, and what is an edge?* Once you can say it, the code is one of two loops.

| The problem says | A node is | An edge is | It asks | Tool |
|---|---|---|---|---|
| a grid of land and water | a cell `(r, c)` | a step up/down/left/right onto land | how many groups? | flood fill (DFS or BFS) |
| "every minute the rot spreads" | a cell | a neighbour, one minute later | when is the last one reached? | multi-source BFS |
| "change one letter at a time" | a word | two words one letter apart | fewest changes | BFS, neighbours from wildcard buckets |
| "slide a tile into the gap" | a whole board, as a string | one slide | fewest moves | BFS over states |
| "keys open locks" | (cell, keys held) | one step | fewest steps | BFS over augmented states |
| "ride buses" | a bus route | two routes share a stop | fewest buses | BFS over routes |
| "deep-copy this graph" | a node object | its neighbour list | a copy | traversal + old→new map |

Weighted edges are Graphs III; "in what order?" and "are these connected?" are Graphs II.

```text
A grid is a graph you never build:              BFS from S spreads in rings of equal distance:

              (r-1, c)                          S . . #            0 1 2 #
    (r, c-1)   (r, c)   (r, c+1)                . # . .    -->     1 # 3 4
              (r+1, c)                          . . . #            2 3 4 #

each cell's edges go to the 4 cells             queue over time:  [0] [1 1] [2 2] [3 3] [4 4]
around it that are on the board                 ring k leaves the queue before ring k+1 starts

DFS on the same grid runs down one corridor as far as it can, then backs up to the last fork.
Same cells, same cost, but its order says nothing about distance.
```

**Why it is fast:** every node goes into the container at most once (it is marked the moment it goes in) and every edge is looked at once from each end, so a traversal costs O(V + E); on a grid that is O(rows · cols). The brute force in most of these problems starts a fresh search from every cell and re-walks the same region again and again: O((rows · cols)²). One `seen` set shared by all the searches is the whole speed-up.

### From idea to code

**The idea in one sentence:** *put the start in a container and mark it; then repeatedly take one node out, and put in every neighbour you have never seen, marking it as it goes in.* A queue (oldest first) makes that BFS; a stack (newest first) makes it DFS.

| Decision | Traversal answer |
|---|---|
| **State**: what must I remember? | the frontier (a `deque` for BFS, a list used as a stack for DFS) and `seen`; for distances a `dist` dict, which can double as `seen` |
| **Definition**: what exactly does each variable mean? | `seen` = "already put in the container, never put it in again"; `dist[v]` = fewest edges from the source to `v`; the frontier = discovered nodes whose neighbours are not expanded yet |
| **Invariant**: what is true at the end of every step? | BFS: the queue holds at most two distances, the `d`s in front of the `d + 1`s, so nodes come out in order of distance. Both: every node in `seen` is expanded exactly once |
| **Step**: how does one node change the state? | pop a node; for each neighbour that is on the board, passable and unseen: mark it, record its distance, push it |
| **Record**: when is the answer updated? | a count goes up when the outer loop starts a *new* flood; a distance is written when a node is *discovered*; the target is checked when a node is popped |
| **Init**: starting values | every source in the container and in `seen`, at distance 0 (one source or many) |
| **Return**: what comes back, and for "not found"? | the distance, count or area; `-1` when the queue runs dry before the target appears |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "the four cells around me" | `for dr, dc in DIRS: nr, nc = r + dr, c + dc` |
| "still on the board" | `0 <= nr < rows and 0 <= nc < cols` |
| "never been there" | `(nr, nc) not in seen` (or `not in dist`) |
| "mark it as I discover it" | `seen.add(nxt)` on the line just before `queue.append(nxt)` |
| "the oldest discovery next" (BFS) | `r, c = queue.popleft()` |
| "the newest discovery next" (DFS) | `r, c = stack.pop()` |
| "one ring further out" | `dist[nxt] = dist[cur] + 1` |
| "a whole ring at once" | `for _ in range(len(queue)):` |
| "a new island starts here" | an outer loop over every cell: `if land and not seen: count += 1` |

Step zero is listing a node's neighbours. From an edge list you build an adjacency list once; on a grid you compute them on the fly.

In [ ]:
def build_graph(n, edges, directed=False):
    graph = [[] for _ in range(n)]               # graph[u] = the nodes one edge away from u
    for u, v in edges:
        graph[u].append(v)
        if not directed:
            graph[v].append(u)                   # undirected: store both directions
    return graph


DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))        # down, up, right, left


def grid_neighbours(grid, r, c):
    """The in-bounds cells next to (r, c). A grid is a graph whose edges you never store."""
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            yield nr, nc


print(build_graph(6, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))   # [[1, 2], [0, 3], [0, 3], [1, 2, 4], [3], []]
print(build_graph(3, [(0, 1), (1, 2)], directed=True))            # [[1], [2], []]
print(list(grid_neighbours(["...", "...", "..."], 0, 0)))         # [(1, 0), (0, 1)]  a corner has 2
print(list(grid_neighbours(["...", "...", "..."], 1, 1)))         # [(2, 1), (0, 1), (1, 2), (1, 0)]

**Try it**
- Compare `build_graph(2, [(0, 1)], directed=True)` with `build_graph(2, [(0, 1)])`: `[[1], []]` versus `[[1], [0]]`. In the directed one, node 1 has no way back to 0.
- Delete the `0 <=` part of the row check and print `list(grid_neighbours(["...", "...", "..."], 0, 0))`: `(-1, 0)` shows up. Python reads `grid[-1]` as the *last* row instead of failing, so the top edge would quietly touch the bottom edge.
- Feed 1-indexed edges: `build_graph(3, [(1, 2), (2, 3)])` raises `IndexError` (there is no `graph[3]`). When nodes are numbered from 1, allocate `n + 1` lists.
- An adjacency matrix becomes lists in one line: for `M = [[0, 1, 1], [1, 0, 0], [1, 0, 0]]`, `[[j for j, x in enumerate(row) if x] for row in M]` gives `[[1, 2], [0], [0]]`.

The BFS template. The tags are the seven decisions:

In [ ]:
def shortest_steps(grid, start, target):
    """Fewest moves from start to target over cells that are not '#'; -1 if unreachable."""
    rows, cols = len(grid), len(grid[0])
    queue = deque([start])                        # STATE: the frontier, oldest first
    dist = {start: 0}                             # STATE + INIT: dist[cell] = fewest moves; also "seen"
    while queue:
        r, c = queue.popleft()                    # STEP: the closest cell not expanded yet
        if (r, c) == target:
            return dist[(r, c)]                   # RETURN: first time it leaves the queue = shortest
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1     # RECORD on discovery: one ring further out
                queue.append((nr, nc))
    return -1                                     # RETURN: the queue ran dry, target never reached


maze = ["S..#",
        ".#..",
        "...#"]
print(shortest_steps(maze, (0, 0), (1, 3)))            # 4
print(shortest_steps(maze, (0, 0), (2, 2)))            # 4
print(shortest_steps(["S#.", "##."], (0, 0), (1, 2)))  # -1

**Try it**
- Swap `queue.popleft()` for `queue.pop()` (a stack) and run `shortest_steps(["....", "....", "...."], (0, 0), (2, 0))`: it says 8 where BFS says 2. Without the queue's order, "first time out = shortest" stops being true.
- Print `[dist[x] for x in queue]` as the first line inside the loop: the queue never holds more than two distances, `d` and `d + 1`. That is the invariant that makes BFS correct.
- Move the target test to discovery time (return right after the `append` when `(nr, nc) == target`): same answers one ring earlier, except `shortest_steps(["S"], (0, 0), (0, 0))` now returns -1. The start is never "discovered", so it needs its own check.
- Predict `shortest_steps(maze, (2, 0), (1, 3))` before running it (4).

The DFS template: count the islands and measure the biggest. The outer loop is what turns "flood one region" into "count the regions":

In [ ]:
def islands(grid):
    """(number of islands, largest island area) for a grid of '1' land and '0' water."""
    rows, cols = len(grid), len(grid[0])
    seen = set()                                  # STATE: cells already pushed (never push twice)
    count = best = 0                              # INIT
    for r in range(rows):
        for c in range(cols):                     # every cell gets a chance to start a flood
            if grid[r][c] != "1" or (r, c) in seen:
                continue
            count += 1                            # RECORD: unseen land = a brand-new island
            seen.add((r, c))
            stack, area = [(r, c)], 0
            while stack:                          # STEP: flood the island with an explicit stack
                x, y = stack.pop()
                area += 1                         # every cell is popped exactly once
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < rows and 0 <= ny < cols and grid[nx][ny] == "1" and (nx, ny) not in seen:
                        seen.add((nx, ny))        # mark on push
                        stack.append((nx, ny))
            best = max(best, area)                # RECORD: the island is complete
    return count, best                            # RETURN


def area_recursive(grid, r, c, seen):
    """The same flood, recursive: check when you ARRIVE, then visit the 4 neighbours."""
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != "1" or (r, c) in seen:
        return 0                                  # off the board, water, or counted already
    seen.add((r, c))
    return 1 + sum(area_recursive(grid, r + dr, c + dc, seen) for dr, dc in DIRS)


sea = ["11000",
       "11000",
       "00100",
       "00011"]
print(islands(sea))                       # (3, 4)
print(area_recursive(sea, 0, 0, set()))   # 4

**Try it**
- Mark at pop time instead: delete `seen.add((nx, ny))` and put `seen.add((x, y))` right after `x, y = stack.pop()`. `islands(["11", "11"])` now says `(1, 5)`: cell (1, 1) was pushed by two neighbours and popped twice.
- Delete `0 <=` from `0 <= nx < rows` and run `islands(["1", "0", "1"])`: `(2, 2)`. The top cell's "up" neighbour is `grid[-1]`, the bottom row, so the first island counts a phantom second cell.
- Run `area_recursive(["1" * 5000], 0, 0, set())`: `RecursionError`. The iterative `islands(["1" * 5000])` answers `(1, 5000)` without trouble.
- Collect every island's `area` in a list instead of keeping the max: `sea` gives `[1, 2, 4]` once sorted.

### Watch it work

Each line is one ring of the BFS; at the end the grid shows every cell's distance from `S`.

In [ ]:
def trace_rings(grid, start):
    dist, queue, ring = {start: 0}, deque([start]), 0
    while queue:
        layer = [queue.popleft() for _ in range(len(queue))]   # exactly the cells at distance `ring`
        print(f"ring {ring}: {layer}")
        for r, c in layer:
            for nr, nc in grid_neighbours(grid, r, c):
                if grid[nr][nc] != "#" and (nr, nc) not in dist:
                    dist[(nr, nc)] = ring + 1
                    queue.append((nr, nc))
        ring += 1
    for r, row in enumerate(grid):
        print("   ", " ".join("#" if ch == "#" else str(dist.get((r, c), "?")) for c, ch in enumerate(row)))


trace_rings(maze, (0, 0))

**Try it**
- Predict the distances for `trace_rings(maze, (1, 2))`, then run it: the rings spread both ways and the far cell (1, 0) is 4 away.
- Wall in a cell: `trace_rings(["S..#", ".##.", "...#"], (0, 0))`. Cell (1, 3) keeps its `?`: that is the `-1` case of `shortest_steps`.
- Pop one cell per iteration instead of a whole layer and print each popped cell's distance: `0 1 1 2 2 3 3 4 4`, never decreasing.

### Where it goes wrong

1. **Marking `seen` when you pop instead of when you push.** A node gets pushed by several neighbours before its first pop: the queue fills with copies and anything counted per pop is counted twice (a 2×2 block of land has area 5). Mark on push.
2. **Half a bounds check.** Without `0 <=`, Python reads `grid[-1]` as the last row instead of failing, so the top edge silently touches the bottom edge. Always write `0 <= nr < rows`.
3. **One search from node 0 on a disconnected graph.** It sees one component only. Counting or visiting everything needs the outer loop that tries every node as a start.
4. **DFS for "fewest steps".** DFS finds *a* path, not the shortest: on an open 3×4 grid a stack-based search reports 8 moves from (0, 0) to (2, 0), BFS reports 2.
5. **Deep recursion.** A recursive flood of a 300×300 island can need 90,000 nested calls; Python stops at about 1,000. Use an explicit stack (or BFS) on big grids, and say so.
6. **Flood fill with the new colour equal to the old one.** If painting is your "seen" mark and the colour doesn't change, nothing is ever marked and the fill never ends. Return early when they are equal.
7. **An extra minute in layered BFS.** Rotting Oranges with `while queue:` counts a minute for the last ring, which rots nothing (`[[2, 1]]` gives 2 instead of 1). Loop `while queue and fresh`.
8. **An undirected edge stored once.** `graph[u].append(v)` without `graph[v].append(u)` leaves `v` with no way back.
9. **`seen` keyed on less than the whole state.** In BFS over states, `(cell, keys)` is the node, not `cell`. Keying on the cell blocks the walk from coming back with more keys.

### Edge cases to say out loud

Start equals target · start or target is a wall · target unreachable · no land at all · one big island · diagonal cells (not neighbours unless the problem says 8 directions) · nodes with no edges · nodes numbered from 1.

In [ ]:
assert shortest_steps(["S"], (0, 0), (0, 0)) == 0            # the start is the target
assert shortest_steps(["S#."], (0, 0), (0, 2)) == -1         # a wall cuts the only way
assert islands(["000"]) == (0, 0)                             # no land: 0 islands, area 0
assert islands(["111", "111"]) == (1, 6)                      # one island covers everything
assert islands(["1", "0", "1"]) == (2, 1)                     # top and bottom rows do not touch
assert islands(["101", "010", "101"]) == (5, 1)               # diagonal cells are not neighbours
assert build_graph(3, []) == [[], [], []]                     # nodes without edges still exist
print("edge cases pass")

**Try it**
- Predict, then add: `assert shortest_steps(["S.", ".."], (0, 0), (1, 1)) == 2`.
- What should `islands(["1"])` return? Write the assert before running it (`(1, 1)`).
- Add the four diagonals to `DIRS` and rerun `islands(["101", "010", "101"])`: one island of 5. Put `DIRS` back afterwards, because the cells below use it.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Count / measure regions** | outer loop over every cell; count the launches; the flood returns its size | 200, 695, 733 |
| **Multi-source BFS** | push *every* source before the loop; a ring = one minute, or one step from the nearest source | 994, 286, 542 |
| **Border-first** | flood from the border instead of asking every cell; what stays unmarked is enclosed | 130, 1020, 417 |
| **Copy a graph** | an `old -> new` dict is both `seen` and the lookup for neighbours | 133 |
| **Implicit neighbours** | a function generates neighbours: wildcard buckets, blank swaps, useful swaps only | 127, 773, 854 |
| **All shortest paths** | BFS layer by layer, storing *every* parent from the previous layer; walk back from the target | 126 |
| **BFS over augmented states** | node = (cell, extra); `seen` keys on the whole tuple | 864, 1293, 847 |
| **BFS over groups** | expand a whole group (a bus route, all equal values) once, then never again | 815, 1345 |

**Multi-source BFS.** All the rotten oranges start spreading at the same moment, so they all go into the queue before the loop starts. Each ring is then one minute, and a cell's ring number is its distance to the *nearest* source. Walls and Gates and 01 Matrix are the same code with `dist[next] = dist[cur] + 1` in place of the minute counter.

In [ ]:
def oranges_rotting(grid):
    grid = [row[:] for row in grid]               # work on a copy
    queue = deque((r, c) for r, row in enumerate(grid) for c, x in enumerate(row) if x == 2)   # ALL sources
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while queue and fresh:                        # stop the moment nothing is fresh
        for _ in range(len(queue)):               # one ring = one minute
            r, c = queue.popleft()
            for nr, nc in grid_neighbours(grid, r, c):
                if grid[nr][nc] == 1:
                    grid[nr][nc] = 2              # rotting it is also marking it seen
                    fresh -= 1
                    queue.append((nr, nc))
        minutes += 1
    return minutes if fresh == 0 else -1


print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))   # 4
print(oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))   # -1  (the bottom-left orange is walled off)
print(oranges_rotting([[2, 1, 1, 1, 2]]))                    # 2   (two waves meet in the middle)

**Try it**
- Change `while queue and fresh:` to `while queue:` and rerun the first grid: 5 instead of 4. The last ring rots nothing but still costs a minute.
- Seed one source only, `queue = deque([(0, 0)])`, and run `[[2, 1, 1, 1, 2]]`: 3 instead of 2. The two waves must start together.
- `oranges_rotting([[0, 2]])` is 0 (nothing to rot) and `oranges_rotting([[1]])` is -1 (nothing to rot it with).
- Turn it into 01 Matrix: seed every 0, keep a `dist` grid instead of `minutes`, and write `dist[nr][nc] = dist[r][c] + 1` when a cell is discovered.

**Border-first.** "Can this cell reach the border?" asked for every cell repeats the same search. Ask the reverse once: *which cells can the border reach?* One flood from all border cells marks everything that escapes; what is left is enclosed. In Pacific Atlantic, water flows downhill, so from the ocean we walk *uphill*. The helper takes the step rule as a function, because that rule is the only thing that changes.

In [ ]:
def flood(grid, starts, can_step):
    """Every cell reachable from any start, stepping from a to b only when can_step(a, b)."""
    seen, stack = set(starts), list(starts)
    while stack:
        r, c = stack.pop()
        for nr, nc in grid_neighbours(grid, r, c):
            if (nr, nc) not in seen and can_step((r, c), (nr, nc)):
                seen.add((nr, nc))
                stack.append((nr, nc))
    return seen


def capture_surrounded(board):
    rows, cols = len(board), len(board[0])
    edge_o = [(r, c) for r in range(rows) for c in range(cols)
              if board[r][c] == "O" and (r in (0, rows - 1) or c in (0, cols - 1))]
    safe = flood(board, edge_o, lambda a, b: board[b[0]][b[1]] == "O")   # O's that touch the border
    return ["".join("O" if (r, c) in safe else "X" for c in range(cols)) for r in range(rows)]


def pacific_atlantic(h):
    rows, cols = len(h), len(h[0])
    uphill = lambda a, b: h[b[0]][b[1]] >= h[a[0]][a[1]]                # water flows down, so we walk up
    pacific = flood(h, [(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)], uphill)
    atlantic = flood(h, [(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)], uphill)
    return sorted(pacific & atlantic)


print(capture_surrounded(["XXXX", "XOOX", "XXOX", "XOXX"]))   # ['XXXX', 'XXXX', 'XXXX', 'XOXX']
heights = [[1, 2, 2, 3, 5],
           [3, 2, 3, 4, 4],
           [2, 4, 5, 3, 1],
           [6, 7, 1, 4, 5],
           [5, 1, 1, 2, 4]]
print(pacific_atlantic(heights))   # [(0, 4), (1, 3), (1, 4), (2, 2), (3, 0), (3, 1), (4, 0)]

**Try it**
- Change `>=` to `>` in `uphill`: `(1, 4)` disappears from the answer. Its only way to the Pacific crosses `(1, 3)`, which has the same height.
- Print `len(pacific), len(atlantic)` inside `pacific_atlantic`: 16 and 16, with 7 cells in common.
- Number of Enclaves with the same helper: for `g = [[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]`, flood from the border land cells with `lambda a, b: g[b[0]][b[1]] == 1`, then subtract the number reached from the total land: 3.
- `capture_surrounded(["XOX", "XOX", "XXX"])` captures nothing: that region touches the top edge.

**Copy a graph.** The copy needs one new node per old node, and every edge must connect *copies*. A dict `old -> new` answers "do I already have a copy of this node?" in O(1), which also makes it the visited set: that is what stops the traversal from going round a cycle forever.

In [ ]:
class Node:
    def __init__(self, val):
        self.val = val
        self.neighbors = []


def clone_graph(node):
    if node is None:
        return None
    clones = {node: Node(node.val)}               # old -> new: the visited set AND the lookup table
    queue = deque([node])
    while queue:
        cur = queue.popleft()
        for nb in cur.neighbors:
            if nb not in clones:                  # first sighting: make its copy, expand it later
                clones[nb] = Node(nb.val)
                queue.append(nb)
            clones[cur].neighbors.append(clones[nb])   # wire copy -> copy, never copy -> original
    return clones[node]


a, b, c, d = Node(1), Node(2), Node(3), Node(4)             # the square 1-2-3-4-1
a.neighbors, b.neighbors, c.neighbors, d.neighbors = [b, d], [a, c], [b, d], [a, c]
copy = clone_graph(a)
print(copy.val, [n.val for n in copy.neighbors])           # 1 [2, 4]
print(copy is a, copy.neighbors[0] is b)                   # False False
print(copy.neighbors[0].neighbors[0] is copy)              # True  (the cycle closes on the copy)

**Try it**
- In the append line, use `nb` instead of `clones[nb]`: now `copy.neighbors[0] is b` prints True. The copy points back into the original graph.
- Delete `queue.append(nb)`: `copy.neighbors[0].neighbors` comes out empty. The copies get made, but only the start is ever expanded.
- Return `clones` instead of `clones[node]` and check its `len`: 4 copies, although every node was reached twice.
- `clone_graph(None)` is `None`, and a lone `Node(7)` copies to a node with no neighbours.

**BFS over states.** The loop never changes; only the meaning of "a node" and the way to list its neighbours. So write the loop once and pass the neighbours as a function. A board becomes a string (hashable, so it fits in `seen`); a word's neighbours come from buckets like `"h*t"` instead of comparing every pair of words; K-Similar Strings only branches on swaps that fix the first wrong letter.

In [ ]:
def bfs_states(starts, neighbours, is_goal):
    """Fewest moves from any start state to a goal state, or -1. States must be hashable."""
    queue = deque((s, 0) for s in starts)
    seen = set(starts)                            # keyed on the WHOLE state
    while queue:
        state, moves = queue.popleft()
        if is_goal(state):
            return moves
        for nxt in neighbours(state):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, moves + 1))
    return -1


TOUCH = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5), 3: (0, 4), 4: (1, 3, 5), 5: (2, 4)}   # 2x3 board, by index


def sliding_puzzle(board):
    def slides(s):                                # swap the blank with each tile touching it
        z = s.index("0")
        for j in TOUCH[z]:
            t = list(s)
            t[z], t[j] = t[j], t[z]
            yield "".join(t)
    start = "".join(str(x) for row in board for x in row)       # a board becomes a hashable string
    return bfs_states([start], slides, lambda s: s == "123450")


def ladder_length(begin, end, words):
    buckets = defaultdict(list)                   # "h*t" -> every word that fits the pattern
    for w in words:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    one_letter_away = lambda w: (v for i in range(len(w)) for v in buckets[w[:i] + "*" + w[i + 1:]])
    moves = bfs_states([begin], one_letter_away, lambda w: w == end)
    return moves + 1 if moves != -1 else 0        # the problem counts words, not moves


def k_similar(s1, s2):
    def fix_first_mismatch(s):                    # only swaps that put the right letter at the first bad slot
        i = next(k for k in range(len(s)) if s[k] != s2[k])
        for j in range(i + 1, len(s)):
            if s[j] == s2[i] and s[j] != s2[j]:
                yield s[:i] + s[j] + s[i + 1:j] + s[i] + s[j + 1:]
    return bfs_states([s1], fix_first_mismatch, lambda s: s == s2)


print(sliding_puzzle([[1, 2, 3], [4, 0, 5]]), sliding_puzzle([[4, 1, 2], [5, 0, 3]]),
      sliding_puzzle([[1, 2, 3], [5, 4, 0]]))                                        # 1 5 -1
print(ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]))     # 5
print(k_similar("abac", "baca"), k_similar("abc", "bca"))                            # 2 2

**Try it**
- Add `print(state, moves)` right after the `popleft()` and run the ladder: `hit 0`, `hot 1`, `dot 2`, `lot 2`, `dog 3`, `log 3`, `cog 4`. Rings, exactly like on the grid.
- Print `len(seen)` just before `return -1` and run the unsolvable board `[[1, 2, 3], [5, 4, 0]]`: 360. Only half of the 720 boards can be reached from it.
- `ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"])` is 0: `"cog"` is in no bucket, so it is never generated.
- Replace `fix_first_mismatch` with a generator of every swap of two different letters: `k_similar("abcdef", "bcdefa")` still says 5, but BFS now pops 720 strings instead of 6.

**BFS over augmented states.** When the same cell can be in different situations, the situation is part of the node: (cell, keys held), (cell, eliminations left), (node, set of visited nodes as a bitmask). Every move still costs one step, so BFS still works; `seen` just has to remember the whole tuple.

In [ ]:
def all_keys(grid):
    rows, cols = len(grid), len(grid[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == "@")
    full = (1 << sum(ch.islower() for row in grid for ch in row)) - 1   # one bit per key

    def moves(state):
        r, c, keys = state
        for nr, nc in grid_neighbours(grid, r, c):
            ch = grid[nr][nc]
            if ch == "#" or (ch.isupper() and not keys & (1 << (ord(ch) - ord("A")))):
                continue                          # a wall, or a lock whose key we do not hold
            yield nr, nc, (keys | (1 << (ord(ch) - ord("a"))) if ch.islower() else keys)
    return bfs_states([(*start, 0)], moves, lambda s: s[2] == full)


def eliminate_obstacles(grid, k):
    rows, cols = len(grid), len(grid[0])

    def moves(state):
        r, c, left = state
        for nr, nc in grid_neighbours(grid, r, c):
            if left - grid[nr][nc] >= 0:          # stepping onto a 1 spends one elimination
                yield nr, nc, left - grid[nr][nc]
    return bfs_states([(0, 0, k)], moves, lambda s: (s[0], s[1]) == (rows - 1, cols - 1))


def visit_all_nodes(graph):
    full = (1 << len(graph)) - 1
    starts = [(i, 1 << i) for i in range(len(graph))]     # start anywhere: every node is a source
    walk = lambda s: ((v, s[1] | (1 << v)) for v in graph[s[0]])
    return bfs_states(starts, walk, lambda s: s[1] == full)


print(all_keys(["@.a..", "###.#", "b.A.B"]), all_keys(["@..aA", "..B#.", "....b"]), all_keys(["@Aa"]))   # 8 6 -1
print(eliminate_obstacles([[0, 0, 0], [1, 1, 0], [0, 0, 0], [0, 1, 1], [0, 0, 0]], 1))                 # 6
print(visit_all_nodes([[1, 2, 3], [0], [0], [0]]))                                                         # 4

**Try it**
- Make `seen` remember only the cell (store `state[:2]` in `bfs_states`) and run `all_keys(["a.@A.b"])`: -1 instead of 7. After fetching `a` on the left, the walk must cross cells it has already seen to reach the lock.
- Run `eliminate_obstacles` on the same grid with `k = 0`: 10, the long way round. With `k = 1` it breaks through one wall and needs 6.
- In `visit_all_nodes`, start from node 0 only (`starts = [(0, 1)]`): 5 instead of 4. Starting at a leaf is shorter, which is why every node is a source.
- Count the states before you run: R × C cells and k keys give at most R · C · 2^k. For `["@.a..", "###.#", "b.A.B"]` that is 15 · 4 = 60.

**BFS over groups.** Sometimes one move reaches a whole group at once: every stop of a bus route, every index holding the same value. The group is the expensive part, so expand each group *once*. Bus Routes counts buses, so the nodes are routes; Jump Game IV deletes a value's bucket the first time it is used.

In [ ]:
def num_buses(routes, source, target):
    if source == target:
        return 0
    routes_at = defaultdict(list)                 # stop -> every route through it
    for i, route in enumerate(routes):
        for stop in route:
            routes_at[stop].append(i)
    queue = deque((i, 1) for i in routes_at[source])   # boarding any bus at source = 1 bus
    boarded, expanded = set(routes_at[source]), {source}
    while queue:
        i, buses = queue.popleft()
        for stop in routes[i]:
            if stop == target:
                return buses
            if stop in expanded:
                continue
            expanded.add(stop)                    # look at each stop's routes once
            for j in routes_at[stop]:
                if j not in boarded:
                    boarded.add(j)                # board each route once
                    queue.append((j, buses + 1))
    return -1


def min_jumps(arr):
    same = defaultdict(list)                      # value -> every index holding it
    for i, x in enumerate(arr):
        same[x].append(i)
    dist, queue = {0: 0}, deque([0])
    while queue:
        i = queue.popleft()
        if i == len(arr) - 1:
            return dist[i]
        for j in [i - 1, i + 1] + same.pop(arr[i], []):   # a value's bucket is used once, then gone
            if 0 <= j < len(arr) and j not in dist:
                dist[j] = dist[i] + 1
                queue.append(j)


print(num_buses([[1, 2, 7], [3, 6, 7]], 1, 6))                                 # 2
print(num_buses([[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]], 15, 12))    # -1
print(min_jumps([100, -23, -23, 404, 100, 23, 23, 23, 3, 404]), min_jumps([7]), min_jumps([7, 6, 9, 6, 9, 6, 9, 7]))   # 3 0 1

**Try it**
- In `min_jumps`, keep the buckets (`same[arr[i]]` instead of `same.pop(arr[i], [])`) and time both on `[7] * 10_000 + [8]`. Both answer 2, but the kept-bucket version re-reads 10,000 equal indexes on every pop and takes seconds instead of milliseconds.
- Print `i, buses` after each `popleft()` in `num_buses` for the first example: route 0 with 1 bus, then route 1 with 2 buses. Stops are never counted.
- `num_buses([[1, 2, 7], [3, 6, 7]], 1, 1)` is 0: you are already there.
- `min_jumps([7, 6, 9, 6, 9, 6, 9, 7])` is 1: the first and last values are equal, so one jump does it.

**All shortest paths (Word Ladder II), in words:** run the BFS layer by layer and record, for every word, *all* the words of the previous layer that reach it (a parents map). Remove a layer's words from the dictionary only after the whole layer is done, so that two parents in the same layer can both register. Stop after the layer that contains `endWord`, then walk the parents map backwards from `endWord` to list every path.

### Say it in the interview

> "Each cell is a node with edges to its four neighbours, so this is a graph search. The brute force starts a new search from every cell and re-walks the same region again and again, O((mn)²). With one shared `seen` set every cell is visited once: O(mn) time and space. Every move costs 1 and we want the fewest moves, so I'll use BFS: the queue hands out cells in order of distance, so the first time the target comes out, its distance is the answer. I mark cells when I enqueue them, so nothing is queued twice."

Then point at three lines: the `seen.add` right before the `append` (mark on push), the bounds check with both halves, and the line where you record. For a state search, say out loud what one state is before writing any code.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| 01 Matrix | `graphs/zero_one_matrix.py` | multi-source BFS from every 0; a cell's first visit is its distance |
| Bus Routes | `graphs/bus_routes.py` | BFS over routes, not stops: a stop → routes index; board each route once, expand each stop once |
| Clone Graph | `graphs/clone_graph.py` · `practice/simple/42_clone_graph.py` | one traversal; the old → new dict is both the visited set and the neighbour lookup |
| Flood Fill | `graphs/flood_fill.py` | recolouring is the visited mark; return early if the new colour equals the old one |
| Jump Game IV | `graphs/jump_game_iv.py` | BFS on indexes; a value's bucket of equal indexes is used once, then deleted |
| K-Similar Strings | `graphs/k_similar_strings.py` | BFS over strings; branch only on swaps that fix the first mismatched slot |
| Max Area of Island | `graphs/max_area_of_island.py` | flood each island once from its first cell; the flood returns its size |
| Number of Enclaves | `graphs/number_of_enclaves.py` | sink every land cell the border can reach; count the land left |
| Number of Islands | `graphs/number_of_islands.py` · `practice/simple/41_number_of_islands.py` | count how many times the scan has to start a new flood |
| Pacific Atlantic Water Flow | `graphs/pacific_atlantic_water_flow.py` | climb uphill from each ocean's border; answer = cells both floods reach |
| Rotting Oranges | `graphs/rotting_oranges.py` · `practice/simple/44_rotting_oranges.py` | multi-source BFS from all rotten oranges; one ring = one minute; -1 if fresh ones remain |
| Shortest Path in a Grid with Obstacles Elimination | `graphs/shortest_path_in_a_grid_with_obstacles_elimination.py` | state = (row, col, eliminations left); more budget at the same cell is a different state |
| Shortest Path to Get All Keys | `graphs/shortest_path_to_get_all_keys.py` | state = (row, col, key bitmask); the goal is any cell with the full mask |
| Shortest Path Visiting All Nodes | `graphs/shortest_path_visiting_all_nodes.py` | state = (node, visited mask); seed all n starts; the first full mask wins |
| Sliding Puzzle | `graphs/sliding_puzzle.py` | the board as a 6-char string; neighbours = swap the blank with a touching index |
| Surrounded Regions | `graphs/surrounded_regions.py` | mark the O's reachable from the border as safe; flip every other O |
| Walls and Gates | `graphs/walls_and_gates.py` | multi-source BFS from all gates; a room still at INF is unvisited |
| Word Ladder | `graphs/word_ladder.py` | BFS on words; wildcard buckets ("h*t") list neighbours without comparing all pairs |
| Word Ladder II | `graphs/word_ladder_ii.py` | BFS by layers keeping every parent from the previous layer; backtrack from endWord |

### Self-check

1. Why mark a node as seen when you push it, not when you pop it?
<details><summary>Answer</summary>Between being pushed and being popped, a node can be discovered again by other neighbours. If it is only marked at pop time, every discovery pushes another copy: the queue grows, work repeats, and anything counted per pop (like an area) is counted more than once. Marking on push lets each node into the container exactly once.</details>

2. Why does BFS give the fewest moves while DFS does not?
<details><summary>Answer</summary>The queue only ever holds distances <code>d</code> and <code>d + 1</code>, in that order, so nodes come out in order of distance; when the target first comes out, no shorter path can still be waiting. DFS follows whichever corridor it tried first, so it reaches the target along <em>some</em> path, not necessarily a short one.</details>

3. In Shortest Path to Get All Keys, why can't <code>seen</code> hold just the cell?
<details><summary>Answer</summary>The same cell with more keys is a different situation: locks that were walls are now open. A walk often has to come back over cells it has already crossed, for example to fetch a key behind it. With cell-only <code>seen</code>, <code>["a.@A.b"]</code> answers -1 instead of 7.</details>

4. Rotting Oranges: why put every rotten orange in the queue at the start instead of running a BFS from each one?
<details><summary>Answer</summary>They all spread at the same time, so the minute an orange rots is its distance to the <em>nearest</em> rotten orange. One BFS seeded with all of them computes exactly that in O(mn). One BFS per source would cost O(sources · mn) and still need a minimum over the sources afterwards.</details>

<a id="s18"></a>

## Graphs II: Ordering & Connectivity

> Two questions about how nodes relate. **Order**: *what can I do first?* Keep taking a node that nothing is waiting on (topological sort). **Connectivity**: *are these two in the same group?* Every group has a boss; merging two groups means pointing one boss at the other (union-find).

**Reach for it when** the problem says prerequisites, dependencies, "must come before", build order, an alphabet hidden in sorted words, or "minimum time when tasks run in parallel" (topological sort); or it says merge, same group, connected, "edges arrive one at a time", "the extra edge that makes a cycle", "after each operation, how many groups?", "queries with a weight limit" (union-find). "Which cable, if cut, disconnects the network?" is bridges; "use every ticket exactly once" is an Eulerian path.

**In this repo:** `graphs/` (19 of its 45 problems) · bank: `practice/simple/43_course_schedule.py`, `practice/simple/47_redundant_connection.py` · basics: `practice/simple/basics/graphs/02_topological_sort_kahn_and_dfs.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
TOPOLOGICAL SORT (Kahn): keep taking whatever nothing is waiting on

      0 -----> 1            indegree = how many arrows point INTO a node
      |        |
      v        v            start      indegree  0:0  1:1  2:1  3:2    ready [0]
      2 -----> 3            take 0  ->           1:0  2:0  3:2         ready [1, 2]
                            take 1  ->                     3:1         ready [2]
                            take 2  ->                     3:0         ready [3]
                            take 3  ->  order 0 1 2 3

A cycle 1 -> 2 -> 3 -> 1: each waits on another, none ever reaches 0, and "ready" runs dry early.
```

```text
UNION-FIND: every group is a tree, and its root is the group's name

  union(0, 1)   union(2, 3)          union(1, 3): find(1) = 0, find(3) = 2,
                                     so hang root 2 under root 0
     0     2                              0
     |     |                             / \
     1     3                            1   2
                                            |
                                            3      find(3) walks 3 -> 2 -> 0; path halving then
                                                   points 3 at its grandparent, the root
  union(0, 2) now: find(0) == find(2) == 0, so it returns False: they were already connected.
```

**Why it is fast:** the brute-force ordering rescans every edge each round to find a node that is ready: O(V · (V + E)). Kahn keeps one counter per node, and taking a node only changes the counters of *its* successors, so each edge is touched once: O(V + E). For connectivity, re-running a BFS for every "connected?" question costs O(V + E) each time. Groups only ever merge, so a forest of parent pointers answers in nearly O(1) per question, as long as the trees stay flat: path compression (or halving) and union by size keep them flat.

### From idea to code

**The idea in two sentences:** *Order: count each node's unfinished prerequisites, take any node whose count is 0, and taking it lowers its successors' counts.* *Connectivity: `find` walks up to the root that names a group, `union` points one root at the other, and a `union` whose two roots are already equal has just found an edge that closes a cycle.*

| Decision | Topological sort (Kahn) | Union-find |
|---|---|---|
| **State**: what must I remember? | `graph[u]` (u before v), `indegree[v]`, and a queue of ready nodes | `parent[x]`, `size[root]`, and the number of groups if asked |
| **Definition**: what exactly does each variable mean? | `indegree[v]` = prerequisites of `v` not taken yet | `find(x)` = the root of x's tree; same root means same group |
| **Invariant**: what is true at the end of every step? | every node in the queue has all its prerequisites already in `order` | every group is one tree; `r` is a root exactly when `parent[r] == r` |
| **Step**: how does one item change the state? | pop `u`, append it; for each successor `v`: `indegree[v] -= 1`, push `v` when it reaches 0 | `union(a, b)`: find both roots; if they differ, hang the smaller tree under the bigger |
| **Record**: when is the answer updated? | the pop order is the topological order | `union` returns `False`: already connected (cycle edge); `True`: one group fewer |
| **Init**: starting values | the queue holds *every* node with indegree 0, isolated nodes too | `parent = list(range(n))`, `size = [1] * n`, `count = n` |
| **Return**: what comes back, and for "not found"? | `order` if `len(order) == n`; otherwise a cycle: `[]`, `False` or `""` | the count, the redundant edge, or groups bucketed by `find` |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "b must come before a" (pair `[a, b]`) | `graph[b].append(a)` and `indegree[a] += 1` |
| "everything I can do right now" | `deque(v for v in range(n) if indegree[v] == 0)` |
| "u is done, so v waits on one fewer" | `indegree[v] -= 1` |
| "v is free now" | `if indegree[v] == 0: queue.append(v)` |
| "some nodes never got free" | `len(order) < n`: there is a cycle |
| "on the path I am walking right now" (DFS) | `color[v] == GRAY` |
| "who is the boss of x's group?" | `find(x)` |
| "merge the two groups" | `parent[find(b)] = find(a)`: roots, never the nodes themselves |
| "they were already together" | `find(a) == find(b)`, so `union` returns `False` |

Kahn's algorithm, written for Course Schedule II. The tags are the seven decisions:

In [ ]:
def find_order(n, prerequisites):
    """Course Schedule II: pair [a, b] means "take b before a". Returns an order, or [] on a cycle."""
    graph = [[] for _ in range(n)]
    indegree = [0] * n                            # STATE: indegree[v] = prerequisites of v not taken yet
    for a, b in prerequisites:
        graph[b].append(a)                        # edge b -> a: finishing b brings a one step closer
        indegree[a] += 1
    queue = deque(v for v in range(n) if indegree[v] == 0)   # INIT: everything that is ready now
    order = []
    while queue:
        u = queue.popleft()                       # STEP: take any ready course
        order.append(u)                           # RECORD: the pop order is a valid order
        for v in graph[u]:
            indegree[v] -= 1                      # v waits on one course fewer
            if indegree[v] == 0:                  # u was v's last missing prerequisite
                queue.append(v)
    return order if len(order) == n else []       # RETURN: anyone never freed sits on a cycle


print(find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))   # [0, 1, 2, 3]
print(find_order(3, [[0, 1], [1, 2], [2, 0]]))           # []  (0 needs 1, 1 needs 2, 2 needs 0)
print(find_order(3, []))                                 # [0, 1, 2]  (no rules: any order)

**Try it**
- Reverse the edge (`graph[a].append(b)` and `indegree[b] += 1`) and rerun the first example: `[3, 1, 2, 0]`, the order backwards. The second example still returns `[]`, because reversing every edge keeps every cycle. That is how this bug survives Course Schedule I and only shows up in II.
- Print `indegree` right after the building loop: `[0, 1, 1, 2]`. Course 3 waits on two courses.
- Replace `popleft()` with `pop()` (a stack): `[0, 2, 1, 3]`, a different order that is just as valid. Kahn does not care which ready node goes first.
- Predict `find_order(4, [[1, 0], [2, 1], [3, 2], [1, 3]])` before running it: `[]`. Course 0 is taken, but 1, 2 and 3 wait on each other in a circle.

The DFS version needs three colours, not two. A node reached again can be *finished* (it was reached earlier by another path, which is fine) or *on the current path* (a cycle). A node finishes after everything it leads to, so the reversed finishing order is a topological order:

In [ ]:
WHITE, GRAY, BLACK = 0, 1, 2                      # not visited, on the current path, finished


def topo_dfs(n, edges):
    """Edges (u, v) mean u before v. Returns reversed DFS finishing order, or [] on a cycle."""
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
    color, finished = [WHITE] * n, []

    def visit(u):                                 # False means "found a cycle"
        color[u] = GRAY                           # u is on the path we are walking right now
        for v in graph[u]:
            if color[v] == GRAY:                  # stepped back onto our own path: a cycle
                return False
            if color[v] == WHITE and not visit(v):
                return False
        color[u] = BLACK                          # everything reachable from u is finished
        finished.append(u)                        # so u finishes after all of its successors
        return True

    for u in range(n):
        if color[u] == WHITE and not visit(u):
            return []
    return finished[::-1]


print(topo_dfs(6, [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]))   # [5, 4, 2, 3, 1, 0]
print(topo_dfs(3, [(0, 1), (0, 2), (2, 1)]))                           # [0, 2, 1]  (1 is reached twice: no cycle)
print(topo_dfs(3, [(0, 1), (1, 2), (2, 0)]))                           # []

**Try it**
- Use two colours only: change `if color[v] == GRAY` to `if color[v] != WHITE`. The diamond `[(0, 1), (0, 2), (2, 1)]` now comes back `[]`: node 1 was finished, not on the path, but the two look the same.
- Print `finished` before reversing: `[0, 1, 3, 2, 4, 5]` for the first example. The deepest nodes finish first.
- Return `finished` without reversing: every edge now points backwards. It is a valid order for the graph with all arrows flipped.
- Run a 10,000-node chain, `topo_dfs(10_000, [(i, i + 1) for i in range(9_999)])`: `RecursionError`. Kahn has no recursion, which is worth saying in an interview.

Union-find. Path halving makes each node on the way up point at its grandparent; union by size hangs the smaller tree below the bigger one. Together they keep every tree nearly flat:

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))              # STATE + INIT: everyone starts as their own root
        self.size = [1] * n                       # size[root] = members of that group
        self.count = n                            # number of groups

    def find(self, x):                            # DEFINITION: the root that names x's group
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving: skip to the grandparent
            x = self.parent[x]
        return x

    def union(self, a, b):                        # STEP: merge the groups of a and b
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                          # RECORD: already connected (this edge closes a cycle)
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra                       # union by size: the smaller tree hangs below
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.count -= 1                           # two groups became one
        return True


dsu = DSU(6)
print([dsu.union(a, b) for a, b in [(0, 1), (2, 3), (1, 3), (0, 2)]])   # [True, True, True, False]
print(dsu.count, dsu.find(3) == dsu.find(0), dsu.size[dsu.find(0)])      # 3 True 4

**Try it**
- On a fresh `DSU(6)` do the unions `(0, 1)`, `(2, 3)`, `(1, 3)`, print `parent`, call `find(3)`, and print again: `[0, 0, 0, 2, 4, 5]` becomes `[0, 0, 0, 0, 4, 5]`. Node 3 now points at its old grandparent, the root.
- Link nodes instead of roots: change `self.parent[rb] = ra` to `self.parent[b] = a`. Then `d = DSU(3); d.union(0, 1); d.union(2, 1); print(d.find(0) == d.find(1))` prints False: node 1 was pulled out from under 0.
- Print `dsu.size`: `[4, 1, 2, 1, 1, 1]`. Only a root's entry means something; `size[2] = 2` is left over from when 2 was a root.
- Delete the two `if ra == rb` lines and rerun the cell: the last union returns True and `count` drops to 2, although the groups are still `{0, 1, 2, 3}`, `{4}` and `{5}`.

### Watch it work

Kahn on the picture's graph, then union-find hunting for the extra edge in Redundant Connection:

In [ ]:
def trace_kahn(n, edges):                         # edges (u, v): u before v
    graph, indegree = [[] for _ in range(n)], [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(v for v in range(n) if indegree[v] == 0)
    print(f"start:  indegree={indegree}  queue={list(queue)}")
    while queue:
        u = queue.popleft()
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
        print(f"take {u}: indegree={indegree}  queue={list(queue)}")


def trace_redundant(edges):                       # nodes are 1..n
    dsu = DSU(len(edges) + 1)
    for u, v in edges:
        merged = dsu.union(u, v)
        print(f"edge {u}-{v}: {'merge' if merged else 'already connected -> redundant'}  parent[1:]={dsu.parent[1:]}")
        if not merged:
            return [u, v]


trace_kahn(4, [(0, 1), (0, 2), (1, 3), (2, 3)])
print(trace_redundant([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))   # [1, 4]

**Try it**
- `trace_kahn(3, [(0, 1), (1, 2), (2, 1)])`: after taking 0 the queue is empty while 1 and 2 still wait on each other. That is what a cycle looks like from inside Kahn's algorithm.
- `trace_redundant([[1, 2], [1, 3], [2, 3]])` returns `[2, 3]`: the triangle closes on its last edge.
- Shuffle the input: `trace_redundant([[1, 4], [3, 4], [1, 3], [1, 2], [4, 5]])` returns `[1, 3]`. The answer is whichever edge of the cycle comes last.

### Where it goes wrong

1. **Edge direction.** `[a, b]` means "b before a", so the edge is `b -> a`. Reversed edges give the order backwards (`[3, 1, 2, 0]` above), and Course Schedule I still passes because reversing edges keeps cycles, so the bug hides until part II.
2. **Forgotten nodes.** Seed the queue with *every* node of indegree 0, including nodes with no edges at all. In Alien Dictionary every letter is a node, even one that appears in no rule (`["ab", "adc"]` must output `c` too).
3. **Duplicate edges with a set adjacency.** If `graph[x]` is a set, add to `indegree[y]` only when `y` was really new; otherwise the count never drops to 0 and a valid order looks like a cycle. With list adjacency duplicates are harmless: counted twice, freed twice.
4. **Two colours in DFS.** "Visited" is not "on my path": with two colours, the diamond `0->1, 0->2, 2->1` looks like a cycle.
5. **Linking nodes instead of roots.** `parent[b] = a` can pull `b` out of its old group. Always link `find(b)` under `find(a)`.
6. **No path compression, no union by size.** The trees can grow into chains of length n, and every `find` then costs O(n).
7. **Counting every union.** Lower the group count only when the roots differed; a repeated edge must not change it.
8. **1-indexed nodes.** Courses, people and tree nodes are often numbered `1..n`: allocate `n + 1`.

### Edge cases to say out loud

One node · no edges (any order is valid, every node is its own group) · a self-loop (a cycle of length 1; `union(x, x)` is False) · duplicate edges · several separate chains · a word that comes before its own prefix (Alien Dictionary) · one node and no edges is a valid tree.

In [ ]:
def respects(order, edges):                       # does every edge (u, v) have u earlier than v?
    pos = {x: i for i, x in enumerate(order)}
    return all(pos[u] < pos[v] for u, v in edges)


assert find_order(1, []) == [0]                                     # one course, nothing to wait for
assert find_order(2, [[1, 1]]) == []                                # a self-loop is a cycle of length 1
assert find_order(3, [[1, 0], [1, 0]]) == [0, 2, 1]                 # a duplicate edge is counted AND freed twice
assert respects(topo_dfs(4, [(0, 1), (2, 3)]), [(0, 1), (2, 3)])    # two separate chains
assert respects(topo_dfs(5, []), [])                                # no edges: any order is valid
d = DSU(3)
assert d.union(0, 0) is False and d.count == 3                      # a node is already with itself
assert d.union(0, 1) and d.union(1, 2) and d.count == 1
print("edge cases pass")

**Try it**
- Predict `find_order(2, [[0, 1], [1, 0]])` before running it (`[]`: two courses waiting on each other).
- `respects([0, 1, 2], [(2, 0)])` is False. Test any order you produce with `respects` instead of comparing it with one "expected" order: many orders are correct.
- `topo_dfs(1, [(0, 0)])` returns `[]`: node 0 is still GRAY when it looks at itself.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Is any order possible?** | count the pops; possible iff all n come out | 207 |
| **Return an order** | keep the pop order | 210 |
| **Derive the edges first** | compare neighbouring words; their first different letter is one edge; prefix check | 269 |
| **Longest path in a DAG** | in topological order, push `finish[v] = max(finish[v], finish[u] + time[v])` | 2050 |
| **An order inside an order** | topo-sort the items and the groups separately; output groups in order, items inside | 1203 |
| **Tree check / component count / judge** | n − 1 edges and no failed union; `dsu.count`; in-degree minus out-degree | 261, 323, 997 |
| **The cycle edge, undirected** | the first union that returns False | 684 |
| **The cycle edge, directed** | a node with two parents first: try dropping the later edge, else the earlier one | 685 |
| **Group by a shared key** | each key (email, prime) remembers its first owner; union with it | 721, 952 |
| **Facts per component** | per root: size and how many infected nodes it holds | 924 |
| **Online counting** | a new node adds 1; each successful union subtracts 1 | 305 |
| **Unions over time** | one timestamp at a time; reset everyone not connected to person 0 | 2092 |
| **Two players** | two DSUs; shared edges first, then each player's own | 1579 |
| **Offline thresholds** | sort queries by limit, let edges in by weight, answer into the original slots | 1697 |
| **Bridges** | DFS low-link: tree edge (u, v) is a bridge iff `low[v] > disc[u]` | 1192 |
| **Use every edge once** | Hierholzer: write a node down when it is stuck, reverse at the end | 332 |

**One call away from the template.** A tree on n nodes is "just enough" edges: exactly n − 1, and none of them wasted on a loop. A component count starts at n and every real merge removes one. The Town Judge needs no graph at all, only degrees.

In [ ]:
def valid_tree(n, edges):                         # exactly n - 1 edges, and none of them closes a cycle
    dsu = DSU(n)
    return len(edges) == n - 1 and all(dsu.union(u, v) for u, v in edges)


def count_components(n, edges):                   # start with n groups; every real merge removes one
    dsu = DSU(n)
    for u, v in edges:
        dsu.union(u, v)
    return dsu.count


def find_judge(n, trust):                         # degrees only: trusted by n - 1 people, trusts nobody
    score = [0] * (n + 1)
    for a, b in trust:
        score[a] -= 1                             # trusting anyone disqualifies a
        score[b] += 1
    return next((i for i in range(1, n + 1) if score[i] == n - 1), -1)


print(valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]]), valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]))   # True False
print(count_components(5, [[0, 1], [1, 2], [3, 4]]))                                                           # 2
print(find_judge(3, [[1, 3], [2, 3]]), find_judge(3, [[1, 3], [2, 3], [3, 1]]), find_judge(1, []))               # 3 -1 1

**Try it**
- Drop the `len(edges) == n - 1` check: `valid_tree(4, [[0, 1], [2, 3]])` becomes True. No cycle, but two pieces; the edge count is what rules that out.
- Delete `score[a] -= 1` in `find_judge`: `find_judge(3, [[1, 3], [2, 3], [3, 1]])` returns 3, a "judge" who trusts somebody.
- `count_components(4, [[0, 1], [1, 0], [2, 3]])` is 2: the repeated edge's union returns False, so the count does not drop.
- `valid_tree(1, [])` is True: a single node with no edges is a tree.

**Derive the edges first (Alien Dictionary).** The sorted word list hides the alphabet. Two *neighbouring* words give at most one fact: at their first different letter, the earlier word's letter comes first. Letters after that difference say nothing. And if a word comes before its own prefix (`"abc"` before `"ab"`), no alphabet can do that.

In [ ]:
def alien_order(words):
    graph = {c: set() for w in words for c in w}  # every letter is a node, even one with no rules
    indegree = {c: 0 for c in graph}
    for w1, w2 in zip(words, words[1:]):          # only NEIGHBOURING words are compared
        for x, y in zip(w1, w2):
            if x != y:                            # the first difference is the only fact: x before y
                if y not in graph[x]:             # a set, so each edge is counted once
                    graph[x].add(y)
                    indegree[y] += 1
                break
        else:                                     # no difference in the common part...
            if len(w1) > len(w2):                 # ...and "abc" sits before "ab": impossible
                return ""
    queue = deque(c for c in graph if indegree[c] == 0)
    order = []
    while queue:
        c = queue.popleft()
        order.append(c)
        for nxt in graph[c]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)
    return "".join(order) if len(order) == len(graph) else ""


print([alien_order(ws) for ws in (["wrt", "wrf", "er", "ett", "rftt"], ["z", "x", "z"], ["abc", "ab"])])   # ['wertf', '', '']

**Try it**
- Delete the `else:` branch (the prefix check): `alien_order(["abc", "ab"])` returns `"abc"` instead of `""`.
- Delete the `break`: `alien_order(["ab", "ba"])` returns `""` instead of `"ab"`, because the letters after the first difference produced the false fact "b before a".
- `alien_order(["ab", "adc"])` is `"abcd"`: `c` appears in no rule but still belongs to the alphabet.
- `alien_order(["z", "z"])` is `"z"`: equal words give no rule and no contradiction.

**Longest path in a DAG (Parallel Courses III).** A course can start only when its *slowest* prerequisite has finished, so `finish[v] = time[v] + max(finish of its prerequisites)`. In Kahn's order every prerequisite has already reported when `v` is popped, so each edge pushes its value forward exactly once.

In [ ]:
def minimum_time(n, relations, time):
    graph, indegree = [[] for _ in range(n + 1)], [0] * (n + 1)
    for a, b in relations:                        # courses are 1..n; a must finish before b starts
        graph[a].append(b)
        indegree[b] += 1
    finish = [0] + time                           # finish[c] = earliest month c can end (so far)
    queue = deque(c for c in range(1, n + 1) if indegree[c] == 0)
    while queue:
        u = queue.popleft()                       # every prerequisite of u has reported: finish[u] is final
        for v in graph[u]:
            finish[v] = max(finish[v], finish[u] + time[v - 1])   # v cannot end before u ends + v's own time
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return max(finish)


print(minimum_time(3, [[1, 3], [2, 3]], [3, 2, 5]))                                 # 8
print(minimum_time(5, [[1, 5], [2, 5], [3, 5], [3, 4], [4, 5]], [1, 2, 3, 4, 5]))   # 12

**Try it**
- Add `time[u - 1]` again (`finish[u] + time[u - 1] + time[v - 1]`): the first example says 11. `finish[u]` already includes u's own time.
- Print `finish` at the end: `[0, 3, 2, 8]`. Course 3 starts at month 3, when its slower prerequisite ends.
- With no relations, `minimum_time(3, [], [3, 2, 5])` is 5: everything runs in parallel.
- Chain them: `minimum_time(3, [[1, 2], [2, 3]], [3, 2, 5])` is 10.

**Group by a shared key.** Comparing every pair of accounts (or numbers) to see whether they share something is O(n²). Turn it around: each key remembers the *first* owner it saw, and every later owner of the same key is unioned with that first one. Accounts share emails; numbers share prime factors.

In [ ]:
def accounts_merge(accounts):
    dsu = DSU(len(accounts))
    owner = {}                                    # email -> the first account that listed it
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in owner:
                dsu.union(i, owner[e])            # a shared email: same person
            else:
                owner[e] = i
    groups = defaultdict(list)
    for e, i in owner.items():
        groups[dsu.find(i)].append(e)             # bucket every email under its group's root
    return sorted([accounts[root][0]] + sorted(es) for root, es in groups.items())


def largest_component(nums):
    dsu = DSU(len(nums))
    owner = {}                                    # prime -> the first index divisible by it
    for i, x in enumerate(nums):
        p = 2
        while p * p <= x:
            if x % p == 0:
                dsu.union(i, owner.setdefault(p, i))
                while x % p == 0:
                    x //= p
            p += 1
        if x > 1:                                 # a prime factor bigger than sqrt is left over
            dsu.union(i, owner.setdefault(x, i))
    return max(dsu.size[dsu.find(i)] for i in range(len(nums)))


print(accounts_merge([["John", "a@m", "b@m"], ["John", "b@m", "c@m"], ["Mary", "d@m"], ["John", "x@m"]]))
# [['John', 'a@m', 'b@m', 'c@m'], ['John', 'x@m'], ['Mary', 'd@m']]
print(largest_component([4, 6, 15, 35]), largest_component([20, 50, 9, 63]))   # 4 2

**Try it**
- Print `owner` and `dsu.parent` after the first loop of `accounts_merge`: `b@m` belongs to account 0, and account 0 now hangs under account 1. The two Johns who share nothing stay apart, although they have the same name.
- `largest_component([2, 3, 6])` is 3: 2 and 3 share nothing, but 6 glues them through the primes 2 and 3.
- Delete the leftover-prime lines (`if x > 1: ...`): `[4, 6, 15, 35]` now answers 2. The 3 in 6 and the 5 in 15 are only found after the loop, so those links never happen.

**Online counting (Number of Islands II).** Land appears one cell at a time and the island count is needed after every step. Recounting the grid each time repeats almost all of the work. Instead: a new cell is a new island (+1), and it swallows each *different* neighbouring island (−1 per successful union).

In [ ]:
def num_islands_2(m, n, positions):
    parent, count, out = {}, 0, []                # only land cells live in the union-find

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r, c in positions:
        if (r, c) not in parent:                  # adding the same cell twice changes nothing
            parent[(r, c)] = (r, c)
            count += 1                            # a new island of one cell...
            for nb in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if nb in parent:                  # (only land cells are keys, so no bounds check)
                    ra, rb = find((r, c)), find(nb)
                    if ra != rb:
                        parent[ra] = rb
                        count -= 1                # ...that swallows each DIFFERENT neighbouring island
        out.append(count)
    return out


print(num_islands_2(3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]]))           # [1, 1, 2, 3]
print(num_islands_2(3, 3, [[0, 0], [0, 2], [1, 1], [0, 1], [0, 1]]))   # [1, 2, 3, 1, 1]

**Try it**
- Remove the duplicate guard (make the `if` always true): `num_islands_2(3, 3, [[0, 0], [0, 0]])` reports `[1, 2]`, a second island that does not exist.
- Decrement for every land neighbour (drop the `if ra != rb`): adding all four cells of a 2×2 grid gives `[1, 1, 1, 0]`. The last cell's two neighbours were already one island.
- Print `{cell: find(cell) for cell in parent}` at the end of the second example: all four cells report the same root.

**Unions over time, and offline thresholds.** Union-find can only merge, never split, so feed it events in the right order. In Find All People With Secret, a timestamp's meetings are unioned together, and afterwards everybody who did *not* end up connected to person 0 is reset to a group of one: their links knew no secret and must not carry one later. In Edge Length Limited Paths, sort the queries by limit and let edges in by weight; each query is then one `find` comparison, written back into its original slot.

In [ ]:
def find_all_people(n, meetings, first):
    dsu = DSU(n)
    dsu.union(0, first)
    meetings = sorted(meetings, key=lambda m: m[2])
    i = 0
    while i < len(meetings):
        j, met = i, set()
        while j < len(meetings) and meetings[j][2] == meetings[i][2]:   # every meeting at this moment
            x, y, _ = meetings[j]
            dsu.union(x, y)
            met.update((x, y))
            j += 1
        for p in met:                             # links that never touched person 0 are forgotten
            if dsu.find(p) != dsu.find(0):
                dsu.parent[p], dsu.size[p] = p, 1
        i = j
    return [p for p in range(n) if dsu.find(p) == dsu.find(0)]


def limited_paths(n, edges, queries):
    dsu = DSU(n)
    edges = sorted(edges, key=lambda e: e[2])
    answer, j = [False] * len(queries), 0
    for qi in sorted(range(len(queries)), key=lambda i: queries[i][2]):   # smallest limit first
        p, q, limit = queries[qi]
        while j < len(edges) and edges[j][2] < limit:   # let in every edge lighter than the limit
            dsu.union(edges[j][0], edges[j][1])
            j += 1
        answer[qi] = dsu.find(p) == dsu.find(q)   # write into the query's ORIGINAL slot
    return answer


print(find_all_people(6, [[1, 2, 5], [2, 3, 8], [1, 5, 10]], 1))   # [0, 1, 2, 3, 5]
print(find_all_people(4, [[2, 3, 1], [1, 2, 2]], 1))               # [0, 1, 2]  (3 met 2 BEFORE 2 knew)
print(limited_paths(3, [[0, 1, 2], [1, 2, 4], [2, 0, 8], [1, 0, 16]], [[0, 1, 2], [0, 2, 5]]))   # [False, True]

**Try it**
- Replace the reset line with `pass`: `find_all_people(4, [[2, 3, 1], [1, 2, 2]], 1)` returns `[0, 1, 2, 3]`. Person 3 "learns" the secret through a meeting that happened before 2 knew it.
- Change `<` to `<=` in `limited_paths`: the first query becomes True, although its only edge weighs 2 and the limit is 2 (the problem says strictly less).
- Ask the queries in the other order, `[[0, 2, 5], [0, 1, 2]]`: `[True, False]`. The answers land in their original slots.
- Print `j` before each answer: how many edges were let in for each limit.

**Bridges (Critical Connections).** Removing each edge and re-running a BFS costs O(E · (V + E)). One DFS can do it: number the nodes in visiting order (`disc`), and let every node report the earliest-numbered node its subtree can reach with one back edge (`low`). A tree edge u → v is a bridge exactly when v's subtree cannot get back to u or above: `low[v] > disc[u]`.

In [ ]:
def critical_connections(n, connections):
    graph = [[] for _ in range(n)]
    for u, v in connections:
        graph[u].append(v)
        graph[v].append(u)
    disc = [-1] * n                               # discovery time; -1 = not visited yet
    low = [0] * n                                 # smallest disc reachable from u's subtree with one back edge
    bridges, timer = [], 0

    def dfs(u, parent):
        nonlocal timer
        disc[u] = low[u] = timer
        timer += 1
        for v in graph[u]:
            if v == parent:
                continue                          # do not count the edge we came in on
            if disc[v] == -1:                     # tree edge: explore the child first
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:              # v's subtree cannot get back to u or above it
                    bridges.append([u, v])
            else:                                 # back edge to a node already on the path
                low[u] = min(low[u], disc[v])

    dfs(0, -1)
    return bridges


print(critical_connections(4, [[0, 1], [1, 2], [2, 0], [1, 3]]))           # [[1, 3]]
print(critical_connections(5, [[0, 1], [1, 2], [2, 0], [1, 3], [3, 4]]))   # [[3, 4], [1, 3]]

**Try it**
- Delete the `if v == parent: continue` lines: the first example returns `[]`. The edge back to the parent looks like a back edge, so every subtree seems to have a way around.
- Change `low[v] > disc[u]` to `>=`: `[0, 1]` is reported too, although it sits on the triangle.
- Print `disc` and `low` after `dfs(0, -1)`: `[0, 1, 2, 3]` and `[0, 0, 0, 3]`. Node 3's low (3) is bigger than node 1's disc (1): nothing below 3 reaches back, so 1-3 is a bridge.
- `critical_connections(3, [[0, 1], [1, 2]])` returns both edges: in a tree every edge is a bridge.

**Use every edge once (Reconstruct Itinerary).** Backtracking over ticket orders can dead-end deep down and undo a lot. Hierholzer's trick: always fly the smallest unused ticket; when you are stuck at an airport, it must be the *end* of what is left, so write it down and back up one step. Airports are written in reverse, so reverse at the end.

In [ ]:
def find_itinerary(tickets):
    graph = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        graph[a].append(b)                        # reverse-sorted, so pop() hands out the smallest
    route, stack = [], ["JFK"]
    while stack:
        while graph[stack[-1]]:                   # keep flying while this airport has unused tickets
            stack.append(graph[stack[-1]].pop())
        route.append(stack.pop())                 # stuck here: this airport is finished, write it down
    return route[::-1]                            # airports were written in reverse finishing order


print(find_itinerary([["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]))   # ['JFK', 'MUC', 'LHR', 'SFO', 'SJC']
print(find_itinerary([["JFK", "KUL"], ["JFK", "NRT"], ["NRT", "JFK"]]))                  # ['JFK', 'NRT', 'JFK', 'KUL']

**Try it**
- Write each airport down on the way *in* instead (append it when you fly to it, no reverse): the second example becomes `['JFK', 'KUL', 'NRT', 'JFK']`, which needs a KUL → NRT ticket that does not exist.
- Sort ascending instead: `pop()` now hands out the *largest* destination. For `[["JFK", "SFO"], ["JFK", "ATL"], ["SFO", "ATL"], ["ATL", "JFK"], ["ATL", "SFO"]]` you get `['JFK', 'SFO', 'ATL', 'JFK', 'ATL', 'SFO']`: it uses every ticket, but it is not the smallest route.
- Print `route[-1], stack` each time an airport is written: KUL goes first, while the stack still holds the trip so far.

**Four more, in words.**
- *Sort Items by Groups (1203):* give every ungrouped item its own new group. Build the item graph and a group graph (`group[u] -> group[v]` when the groups differ), topo-sort both, then output the groups in group order with each group's items in item order. Either sort failing means `[]`.
- *Redundant Connection II (685):* directed, one parent per node. If some node has two parents, the answer is one of those two edges: skip the later one and union the rest; if a cycle still appears, the earlier edge was the culprit. If no node has two parents, return the first edge that closes a cycle.
- *Remove Max Edges to Keep Graph Fully Traversable (1579):* two DSUs, one per player. Shared (type 3) edges go first, kept once if they merge anything; then type 1 for Alice and type 2 for Bob. Answer = edges − kept, or −1 if either player's DSU still has more than one group.
- *Minimize Malware Spread (924):* union the adjacency matrix, count the infected nodes per root, and remove the infected node whose component has exactly one infected node and the largest size (smallest index on ties; `min(initial)` if there is none).

### Say it in the interview

> "Prerequisites form a directed graph, and I need an order that respects every edge: a topological sort. The brute force rescans all edges for a course with nothing pending, O(V · (V + E)). I'll keep an indegree counter per course and a queue of courses at zero; taking a course only decrements its successors, so every edge is touched once, O(V + E). If fewer than n courses come out, the rest are stuck on a cycle."

> "Groups only ever merge, so I'll use union-find with path compression and union by size: nearly O(1) per operation. A union that finds both ends already under the same root is exactly the edge that closes a cycle."

Point at the `indegree[v] == 0` line, the `len(order) == n` check, and the `if ra == rb: return False` line: those three lines are the whole idea.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Accounts Merge | `graphs/accounts_merge.py` | email → first account index; union on every repeat; bucket emails by root |
| Alien Dictionary | `graphs/alien_dictionary.py` | neighbouring words give one edge at their first difference; a word before its prefix → ""; Kahn |
| Checking Existence of Edge Length Limited Paths | `graphs/checking_existence_of_edge_length_limited_paths.py` | offline: sort queries by limit, union the edges lighter than it, answer in the original order |
| Course Schedule | `graphs/course_schedule.py` · `practice/simple/43_course_schedule.py` | Kahn: possible iff all n courses get popped |
| Course Schedule II | `graphs/course_schedule_ii.py` | Kahn's pop order is the answer; shorter than n means a cycle → [] |
| Critical Connections in a Network | `graphs/critical_connections_in_a_network.py` | Tarjan: tree edge (u, v) is a bridge iff low[v] > disc[u] |
| Find All People With Secret | `graphs/find_all_people_with_secret.py` | union one timestamp at a time; reset everyone not connected to person 0 |
| Find the Town Judge | `graphs/find_the_town_judge.py` | score = in-degree − out-degree; only the judge reaches n − 1 |
| Graph Valid Tree | `graphs/graph_valid_tree.py` | exactly n − 1 edges and no union fails |
| Largest Component Size by Common Factor | `graphs/largest_component_size_by_common_factor.py` | union each number with the first owner of each prime factor; count numbers per root |
| Minimize Malware Spread | `graphs/minimize_malware_spread.py` | removing a node saves its component only if it is the only infected one; pick the biggest |
| Number of Connected Components in an Undirected Graph | `graphs/number_of_connected_components.py` | start at n; every successful union subtracts one |
| Number of Islands II | `graphs/number_of_islands_ii.py` | new land: +1, then −1 for each distinct neighbouring root it merges |
| Parallel Courses III | `graphs/parallel_courses_iii.py` | longest path in a DAG: push finish times forward in Kahn order |
| Reconstruct Itinerary | `graphs/reconstruct_itinerary.py` | Hierholzer: fly the smallest ticket, write an airport when stuck, reverse |
| Redundant Connection | `graphs/redundant_connection.py` · `practice/simple/47_redundant_connection.py` | the first union whose two ends already share a root |
| Redundant Connection II | `graphs/redundant_connection_ii.py` | a two-parent node? try dropping the later edge, else the earlier; otherwise the cycle edge |
| Remove Max Number of Edges to Keep Graph Fully Traversable | `graphs/remove_max_number_of_edges_to_keep_graph_fully_traversable.py` | two DSUs; shared edges first; removable = edges − kept |
| Sort Items by Groups Respecting Dependencies | `graphs/sort_items_by_groups_respecting_dependencies.py` | topo-sort items and groups separately; loners get their own group |

### Self-check

1. You reversed every edge in Course Schedule by mistake. Why does part I still pass while part II fails?
<details><summary>Answer</summary>Reversing every edge turns each cycle into a cycle and each acyclic graph into an acyclic graph, so "is there an order?" has the same answer. But the order you return is valid for the *reversed* graph: every course now comes out before its own prerequisites, so part II's answer is wrong.</details>

2. Kahn's algorithm stopped with `len(order) < n`. What can you say about the nodes left over?
<details><summary>Answer</summary>Each of them still has indegree at least 1, and every unfinished prerequisite of a leftover node is itself a leftover. Walking backwards along those prerequisites never has to stop, so in a finite graph it must repeat a node: there is a cycle among the leftovers. (Some leftovers may only sit downstream of the cycle.)</details>

3. What does `union(a, b)` returning False tell you, and which problems use exactly that?
<details><summary>Answer</summary>a and b were already connected before this edge, so the edge closes a cycle. Redundant Connection returns that edge, Graph Valid Tree fails on it, a component count ignores it, and Kruskal's MST (Graphs III) skips it.</details>

4. Why does Find All People With Secret reset some people after each timestamp?
<details><summary>Answer</summary>A meeting at time 5 between two people who know nothing passes nothing on, but union-find would remember that link forever. If one of them learns the secret at time 8, the old link would wrongly carry it to the other. Resetting everyone not connected to person 0 forgets those empty links; people connected to 0 stay merged, because knowing the secret is permanent.</details>

<a id="s19"></a>

## Graphs III: Weighted Paths & MST

> Once edges have prices, "fewest edges" is no longer "cheapest". **Dijkstra** is BFS with a priority queue: always settle the closest unsettled node, because with no negative edges nobody can reach it more cheaply later. **A minimum spanning tree** connects everything for the least total price: keep taking the cheapest edge that does not close a loop.

**Reach for it when** edges have weights and the problem asks for the minimum cost / time / effort to get somewhere (Dijkstra); every weight is 0 or 1, like "free move or paid move" (0-1 BFS with a deque); "minimise the *highest* step on the route" (Dijkstra with `max` instead of `+`); "connect all points / cities as cheaply as possible" (MST: Prim or Kruskal); "at most k stops" or negative weights (Bellman-Ford rounds).

**In this repo:** `graphs/` (7 of its 45 problems) · bank: `practice/simple/45_network_delay_time.py`, `practice/simple/46_min_cost_to_connect_all_points.py` · basics: `practice/simple/basics/graphs/06_dijkstra.py`, `practice/simple/basics/graphs/05_prim_mst.py`, `practice/simple/basics/graphs/04_kruskal_mst.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
         1
   A ---------- B          pop (0,A)  settle A  push B=1 C=5   heap (1,B) (5,C)
   |            |          pop (1,B)  settle B  push D=3       heap (3,D) (5,C)
 5 |            | 2        pop (3,D)  settle D  push C=4       heap (4,C) (5,C)   C got cheaper
   |            |          pop (4,C)  settle C                 heap (5,C)
   C ---------- D          pop (5,C)  stale: 5 > dist[C] = 4, skip it
         1
```

The heap always hands out the closest node nobody has settled yet. C was first priced at 5 (straight from A), then found again at 4 (A, B, D, C); the old entry is not removed, it is simply skipped when it surfaces. That skip is the **stale check**.

```text
    0 ---4--- 1                 Kruskal: cheapest edge first; skip it if its ends are already connected
    |       / |
    1     2   5                 0-2 (1)  take   {0,2} {1} {3} {4}
    |   /     |                 1-2 (2)  take   {0,1,2} {3} {4}
    2 ---8--- 3 ---3--- 4       3-4 (3)  take   {0,1,2} {3,4}
                                0-1 (4)  SKIP   0 and 1 are already connected: it would close a loop
                                1-3 (5)  take   {0,1,2,3,4}: n - 1 = 4 edges, done, total 11

    Prim grows one blob from 0 instead, always adding the cheapest edge that leaves it:
    0-2 (1), 2-1 (2), 1-3 (5), 3-4 (3): the same total, 11
```

**Why it is fast:** Bellman-Ford relaxes every edge in V − 1 rounds, O(V · E), because it does not know which distances are final. Dijkstra settles nodes in order of distance, so each node's edges are relaxed once and the heap finds the next node in O(log V): O((V + E) log V). Why is a popped node final? Any other route to it must leave the settled region through some node still in the heap, which is already at least as far, and non-negative edges can only add to that. For the MST, the brute force tries spanning trees (exponentially many); the **cut property** says the cheapest edge crossing any split of the nodes is safe to take, so the greedy choice is never wrong.

### From idea to code

**The idea in one sentence:** *pop the cheapest entry from a min-heap; if it is out of date, skip it; otherwise it is final, so offer its neighbours at their new prices.* Dijkstra prices a node by the whole path `d + w`; Prim prices it by the single edge `w` that would attach it to the tree. That is the one real difference between the two loops: both pop the cheapest entry, skip the out-of-date ones, and offer the neighbours.

| Decision | Dijkstra | Prim (MST) |
|---|---|---|
| **State**: what must I remember? | `dist[v]` and a min-heap of `(dist, node)` | `in_tree` and a min-heap of `(edge cost, node)` |
| **Definition**: what exactly does each variable mean? | `dist[v]` = the cheapest path from `src` to `v` found so far; final once `v` is popped with `d == dist[v]` | an entry `(w, v)` = "v could join the tree through an edge of cost w" |
| **Invariant**: what is true at the end of every step? | nodes are settled in nondecreasing distance, and a settled distance never changes | the tree built so far is part of some minimum spanning tree |
| **Step**: how does one item change the state? | pop `(d, u)`; skip if `d > dist[u]`; for each edge `(v, w)`: if `d + w < dist[v]`, update and push | pop `(w, v)`; skip if `v` is in the tree; add it; push `(w', x)` for every edge to an outside `x` |
| **Record**: when is the answer updated? | `dist[u]` is final at its (non-stale) pop | `total += w` when `v` joins |
| **Init**: starting values | `dist = {src: 0}`, heap `[(0, src)]` | heap `[(0, start)]`: the start joins for free |
| **Return**: what comes back, and for "not found"? | `dist`, `dist[target]` or `max(dist.values())`; −1 if a node never appears | `total` once all n have joined; fewer means the graph is disconnected |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "the closest node not settled yet" | `d, u = heapq.heappop(heap)` |
| "an out-of-date copy of u" | `if d > dist[u]: continue` |
| "going through u is cheaper" | `if d + w < dist.get(v, math.inf):` |
| "remember it and offer it" | `dist[v] = d + w` then `heapq.heappush(heap, (d + w, v))` |
| "a path is as bad as its worst step" | `max(d, w)` in place of `d + w` |
| "a free move / a paid move" (0-1 BFS) | `dq.appendleft(...)` / `dq.append(...)` |
| "the cheapest edge leaving the tree" (Prim) | `w, v = heapq.heappop(heap)`, skip if `in_tree[v]` |
| "this edge would close a loop" (Kruskal) | `find(u) == find(v)` |

Dijkstra, then Network Delay Time on top of it. The tags are the seven decisions:

In [ ]:
def dijkstra(graph, src):
    """graph[u] = [(v, w), ...] with every w >= 0. Returns {node: shortest distance from src}."""
    dist = {src: 0}                               # STATE + INIT: dist[v] = cheapest way to v found so far
    heap = [(0, src)]                             # STATE: (distance, node), the smallest on top
    while heap:
        d, u = heapq.heappop(heap)                # STEP: the closest node not settled yet
        if d > dist[u]:
            continue                              # stale entry: u was already settled closer
        for v, w in graph[u]:                     # RECORD: d == dist[u] is final from here on
            if d + w < dist.get(v, math.inf):     # relax: going through u is cheaper
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    return dist                                   # RETURN: unreachable nodes are simply missing


def network_delay_time(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = dijkstra(graph, k)
    return max(dist.values()) if len(dist) == n else -1     # the last node to hear it, or -1


print(network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2))             # 2
print(network_delay_time([[1, 2, 4], [1, 3, 1], [3, 2, 1], [2, 4, 2]], 4, 1))  # 4  (1 -> 3 -> 2 beats 1 -> 2)
print(network_delay_time([[1, 2, 1]], 2, 2))                                   # -1 (nothing leaves node 2)

**Try it**
- Add `print(d, u)` right after the `heappop` and run the second example: node 2 comes out twice, at 2 (real) and at 4 (stale, skipped).
- Make the ticket 1 → 2 cost 1 instead of 4 and predict the answer before running it: 3, along 1 → 2 → 4.
- Print `dist` inside `network_delay_time` for the second example: `{1: 0, 2: 2, 3: 1, 4: 4}`. Every key is a node the signal reaches.

Prim on the complete graph of points. There is no edge list: the "edges" from a point are its distances to every point still outside the tree.

In [ ]:
def min_cost_connect_points(points):
    n = len(points)
    in_tree = [False] * n                         # STATE: who has joined the tree
    heap = [(0, 0)]                               # STATE + INIT: (cost to join, point); point 0 joins free
    total = joined = 0
    while joined < n:
        cost, i = heapq.heappop(heap)             # STEP: the cheapest edge leaving the tree
        if in_tree[i]:
            continue                              # stale: i joined earlier through a cheaper edge
        in_tree[i] = True
        total += cost                             # RECORD: the cheapest crossing edge is always safe
        joined += 1
        for j in range(n):                        # offer i's edges to every point still outside
            if not in_tree[j]:
                manhattan = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
                heapq.heappush(heap, (manhattan, j))
    return total                                  # RETURN


print(min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]))   # 20
print(min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]))                # 18

**Try it**
- Delete the `if in_tree[i]: continue` lines: the first example answers 18. Point 3 is popped a second time and counted as a fifth join, so the loop stops before point 2 ever joins.
- Start the heap with `(5, 0)` instead of `(0, 0)`: 25. The start must join for free.
- Print `i, cost` when a point joins: 0 (0), 1 (4), 3 (3), 4 (4), 2 (9). The costs are not sorted: Prim takes the cheapest edge leaving the *current* tree, not the cheapest edge overall.

Kruskal: sort all edges once, then let union-find say whether an edge would close a loop.

In [ ]:
def kruskal(n, edges):
    """edges = [(u, v, w)]. Returns (total weight, edges kept). A disconnected graph gives a forest."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]         # path halving
            x = parent[x]
        return x

    total, kept = 0, []
    for u, v, w in sorted(edges, key=lambda e: e[2]):   # STEP: cheapest edge first
        ru, rv = find(u), find(v)
        if ru == rv:
            continue                              # both ends already connected: it would close a loop
        parent[ru] = rv
        total += w                                # RECORD
        kept.append((u, v, w))
        if len(kept) == n - 1:
            break                                 # a spanning tree has exactly n - 1 edges
    return total, kept


print(kruskal(5, [(0, 1, 4), (0, 2, 1), (1, 2, 2), (1, 3, 5), (2, 3, 8), (3, 4, 3)]))
# (11, [(0, 2, 1), (1, 2, 2), (3, 4, 3), (1, 3, 5)])

**Try it**
- Delete the `if ru == rv: continue` lines: you get `(10, [(0, 2, 1), (1, 2, 2), (3, 4, 3), (0, 1, 4)])`, four edges that contain the loop 0-1-2 and leave 3 and 4 cut off from the rest.
- Sort by `-e[2]` (heaviest first): a *maximum* spanning tree, weight 20.
- `kruskal(4, [(0, 1, 1), (2, 3, 1)])` keeps only 2 edges: fewer than `n - 1` means the graph is disconnected.

### Watch it work

The picture's graph, step by step. Every pop is either settled or skipped as stale:

In [ ]:
def trace_dijkstra(graph, src):
    dist, heap = {src: 0}, [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            print(f"pop ({d}, {u})  stale: dist[{u}] is already {dist[u]}, skip")
            continue
        pushed = []
        for v, w in graph[u]:
            if d + w < dist.get(v, math.inf):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
                pushed.append(f"{v}={d + w}")
        print(f"pop ({d}, {u})  settle {u}  push {pushed}  heap={sorted(heap)}")


roads = {"A": [("B", 1), ("C", 5)], "B": [("A", 1), ("D", 2)],
         "C": [("A", 5), ("D", 1)], "D": [("B", 2), ("C", 1)]}
trace_dijkstra(roads, "A")

**Try it**
- Make the road A-C cost 3 instead of 5 (in both directions): no stale entry this time, because C's first price is already its best.
- Add `last = max(last, d)` right after the pop, *before* the stale check (start with `last = 0`), and print it at the end: 5, although the farthest node is only 4 away. Anything recorded per pop belongs after the stale check.
- Run `trace_dijkstra({"A": [("B", 1), ("C", 2)], "B": [("D", 1)], "C": [("B", -2)], "D": []}, "A")`: B is settled twice, first at 1 and later at 0. A negative edge broke "popped means final".

### Where it goes wrong

1. **Plain BFS on weighted edges.** BFS counts edges, not cost: in the second delay example it would accept 1 → 2 (one edge, cost 4) and never notice 1 → 3 → 2 (cost 2).
2. **No stale check while recording at pop time.** Distances still come out right (an out-of-date `d` cannot improve anything), but whatever you record per pop is wrong: the max over pops in the trace graph is 5 instead of 4, and Prim counts a point twice.
3. **Negative edges.** "Popped means final" needs `w >= 0`. With `A->B 1, A->C 2, C->B -2, B->D 1`, B is popped at 1 before the cheaper route through C (0) is found. A version that never re-expands a node (a `done` set) leaves D at 2 instead of 1; this lazy version re-pushes B and recovers here, but can take exponential time in general. Negative weights need Bellman-Ford.
4. **0-1 BFS with a plain FIFO.** A free move must go to the *front*; appending it lets a cost-1 entry come out first, and `[[1, 1, 3], [3, 2, 2], [1, 1, 4]]` answers 1 instead of 0.
5. **Bottleneck with the wrong key.** Pushing `grid[nr][nc]` instead of `max(t, grid[nr][nc])` forgets the highest cell on the way (the 5×5 example below answers 6 instead of 16).
6. **Unreachable nodes.** They never get a `dist`. Check `len(dist) == n` before taking a max, and return −1 otherwise.
7. **Prim's start with a cost.** The start joins for free: push `(0, start)`.
8. **Losing edge indexes.** Sorting `edges` in place loses the original numbering that Critical and Pseudo-Critical Edges must return. Sort a list of indexes.
9. **Unorderable heap entries.** On a tie in distance, Python compares the next item of the tuple; two custom objects there raise `TypeError`. Push `(dist, index)` or `(dist, counter, obj)`.

### Edge cases to say out loud

The source alone · an unreachable node (−1) · zero-weight edges · parallel edges (the cheaper one wins) · self-loops (harmless) · one point (MST cost 0) · duplicate points (cost 0) · a disconnected graph for an MST (fewer than n − 1 edges) · equal weights (several MSTs, one total).

In [ ]:
assert dijkstra({0: []}, 0) == {0: 0}                                  # the source alone
assert dijkstra({0: [(1, 0)], 1: []}, 0) == {0: 0, 1: 0}               # zero-weight edges are fine
assert dijkstra({0: [(1, 5), (1, 2)], 1: []}, 0)[1] == 2               # parallel edges: the cheaper one wins
assert network_delay_time([], 1, 1) == 0                                # one node hears it at time 0
assert network_delay_time([[1, 2, 1]], 3, 1) == -1                      # node 3 never hears it
assert min_cost_connect_points([[1, 1]]) == 0                           # one point: nothing to connect
assert min_cost_connect_points([[0, 0], [0, 0]]) == 0                   # duplicate points cost 0
assert kruskal(3, [(0, 1, 1)]) == (1, [(0, 1, 1)])                      # disconnected: fewer than n - 1 edges
print("edge cases pass")

**Try it**
- Predict `dijkstra({0: [(1, 3)], 1: [(0, 3)]}, 1)` before running it: `{1: 0, 0: 3}`.
- `dijkstra({0: [(0, 5)]}, 0)` is still `{0: 0}`: a loop can never make a path cheaper.
- `min_cost_connect_points([[0, 0], [1, 0], [2, 0]])` is 2: three points on a line, and the tree is the line.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **All distances, then a max** | run to the end; answer = `max(dist.values())`, −1 if a node is missing | 743 |
| **Weights are only 0 or 1** | a deque instead of a heap: weight 0 → `appendleft`, weight 1 → `append` | 1368, 1263 |
| **Dijkstra on a grid or on states** | a node is `(r, c)` or a whole state like `(box, player)`; the loop is unchanged | 778, 1263 |
| **Minimise the worst step** | key = `max(so_far, step)` instead of a sum | 778 |
| **Several fixed endpoints** | one Dijkstra per endpoint; reverse the edges for "x → dest"; combine at a meeting node | 2203 |
| **At most k edges, or negative weights** | Bellman-Ford: k rounds, each relaxing every edge from last round's distances | 787 |
| **MST on a set of points** | Prim with a heap (or an O(n²) array scan) on the complete graph | 1584 |
| **MST from an edge list** | Kruskal: sort + union-find | 1489 |
| **Which edges matter to the MST?** | rerun Kruskal without each edge (critical) and with it forced first (pseudo-critical) | 1489 |

**Weights are only 0 or 1: 0-1 BFS.** With two possible weights the heap only ever holds two distances, `d` and `d + 1`. A deque keeps that order for free: a free move joins the *front* of the line, a paid move the back. In Minimum Cost to Make a Valid Path, following a cell's arrow is free and turning it costs 1.

In [ ]:
ARROWS = {1: (0, 1), 2: (0, -1), 3: (1, 0), 4: (-1, 0)}   # sign in the cell -> direction it points


def min_cost_valid_path(grid):
    rows, cols = len(grid), len(grid[0])
    dist = {(0, 0): 0}
    dq = deque([(0, 0, 0)])                         # (cost, r, c): only two cost values live in here, d and d + 1
    while dq:
        d, r, c = dq.popleft()
        if d > dist[(r, c)]:
            continue                                # stale, exactly like Dijkstra
        if (r, c) == (rows - 1, cols - 1):
            return d
        for sign, (dr, dc) in ARROWS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                w = 0 if grid[r][c] == sign else 1  # follow the arrow for free, turn it for 1
                if d + w < dist.get((nr, nc), math.inf):
                    dist[(nr, nc)] = d + w
                    if w == 0:
                        dq.appendleft((d, nr, nc))  # same cost: jump the line
                    else:
                        dq.append((d + 1, nr, nc))  # one more: back of the line


print(min_cost_valid_path([[1, 1, 1, 1], [2, 2, 2, 2], [1, 1, 1, 1], [2, 2, 2, 2]]))   # 3
print(min_cost_valid_path([[1, 1, 3], [3, 2, 2], [1, 1, 4]]))                           # 0
print(min_cost_valid_path([[1, 2], [4, 3]]))                                            # 1

**Try it**
- Change `dq.appendleft(...)` to `dq.append(...)`, a plain FIFO: `[[1, 1, 3], [3, 2, 2], [1, 1, 4]]` now answers 1 instead of 0. A cost-1 entry got out before the free path arrived.
- Print `[x[0] for x in dq]` just before each `popleft()` on the first grid: only two values ever, `d` and `d + 1`, in order. That is why a deque can replace the heap.
- Swap the deque for `heapq` (push `(d + w, nr, nc)`, pop the smallest): plain Dijkstra, the same three answers, with an extra log factor.

The same deque works on states. In Minimum Moves to Move a Box (1263) a node is `(box, player)`: a player step that does not touch the box costs 0, and walking into the box pushes it one cell for a cost of 1 (if the cell beyond is free). The answer is the cost of the first popped state whose box sits on the target.

**Minimise the worst step: bottleneck Dijkstra.** In Swim in Rising Water a path is as hard as its *highest* cell, not its total. Dijkstra still works, because `max` never decreases along a path, just as `+` never decreases with non-negative weights. Only the key changes.

In [ ]:
def swim_in_water(grid):
    n = len(grid)
    level = {(0, 0): grid[0][0]}                    # level[cell] = lowest possible "highest cell" on a path to it
    heap = [(grid[0][0], 0, 0)]
    while heap:
        t, r, c = heapq.heappop(heap)
        if t > level[(r, c)]:
            continue
        if (r, c) == (n - 1, n - 1):
            return t
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                nt = max(t, grid[nr][nc])           # a path is as hard as its highest cell: max, not +
                if nt < level.get((nr, nc), math.inf):
                    level[(nr, nc)] = nt
                    heapq.heappush(heap, (nt, nr, nc))


print(swim_in_water([[0, 2], [1, 3]]))                                                 # 3
print(swim_in_water([[0, 1, 2, 3, 4], [24, 23, 22, 21, 5], [12, 13, 14, 15, 16],
                     [11, 17, 18, 19, 20], [10, 9, 8, 7, 6]]))                         # 16

**Try it**
- Push `grid[nr][nc]` instead of `max(t, grid[nr][nc])`: the 5×5 grid answers 6 instead of 16. The key forgot the highest cell on the way.
- Use `t + grid[nr][nc]`: 57, the cheapest *total* climb, which is a different problem.
- `swim_in_water([[3, 2], [0, 1]])` is 3: the start's own height counts, so you cannot leave before the water reaches it.

**Several fixed endpoints.** Two paths, from `src1` and `src2`, must both reach `dest`. Somewhere they meet at a node `x` and share the rest, so the cost is `d1[x] + d2[x] + (x → dest)`. Distances from the sources are two ordinary Dijkstras. "From every x to dest" sounds like n searches, but on the reversed graph it is a single Dijkstra *from* `dest`.

In [ ]:
def minimum_weight(n, edges, src1, src2, dest):
    forward, backward = defaultdict(list), defaultdict(list)
    for u, v, w in edges:
        forward[u].append((v, w))
        backward[v].append((u, w))                  # reversed: Dijkstra from dest measures every "x -> dest"
    d1, d2, to_dest = dijkstra(forward, src1), dijkstra(forward, src2), dijkstra(backward, dest)
    best = min((d1[x] + d2[x] + to_dest[x] for x in range(n) if x in d1 and x in d2 and x in to_dest),
               default=math.inf)                    # the two paths meet at x, then share x -> dest
    return -1 if best == math.inf else best


print(minimum_weight(6, [[0, 2, 2], [0, 5, 6], [1, 0, 3], [1, 4, 5], [2, 1, 1], [2, 3, 3], [2, 3, 4],
                         [3, 4, 2], [4, 5, 1]], 0, 1, 5))   # 9
print(minimum_weight(3, [[0, 1, 1], [2, 1, 1]], 0, 1, 2))  # -1

**Try it**
- Run the third Dijkstra on `forward` instead of `backward`: 12. Nothing can be reached *from* node 5, so the only meeting point left is 5 itself.
- Print `d1[x]`, `d2[x]` and `to_dest[x]` for every x: meeting at 0 or at 1 both cost 9 (`0 + 3 + 6` and `3 + 0 + 6`).
- `minimum_weight(3, [[0, 2, 5], [1, 2, 5], [0, 1, 1]], 0, 1, 2)` is 6: src1 rides to src2 for 1, then they share 1 → 2. The meeting point can be one of the sources.

**Which edges matter to the MST?** An edge is *critical* if the MST without it is heavier (or impossible), and *pseudo-critical* if it is not critical but some MST uses it: forcing it in first still gives the best weight. Kruskal is cheap, so run it once per question.

In [ ]:
def critical_edges(n, edges):
    order = sorted(range(len(edges)), key=lambda i: edges[i][2])   # sort INDEXES: keep the original numbering

    def mst_weight(skip=None, force=None):
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        total, used = 0, 0
        for i in ([force] if force is not None else []) + order:   # a forced edge goes in first
            u, v, w = edges[i]
            if i == skip or find(u) == find(v):
                continue
            parent[find(u)] = find(v)
            total, used = total + w, used + 1
        return total if used == n - 1 else math.inf  # not spanning counts as infinitely bad

    best = mst_weight()
    critical = [i for i in range(len(edges)) if mst_weight(skip=i) > best]
    pseudo = [i for i in range(len(edges)) if i not in critical and mst_weight(force=i) == best]
    return [critical, pseudo]


print(critical_edges(5, [[0, 1, 1], [1, 2, 1], [2, 3, 2], [0, 3, 2], [0, 4, 3], [3, 4, 3], [1, 4, 6]]))   # [[0, 1], [2, 3, 4, 5]]
print(critical_edges(4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 1]]))                                  # [[], [0, 1, 2, 3]]

**Try it**
- Let a run that does not span count as a normal weight (`return total`): the path `critical_edges(3, [[0, 1, 1], [1, 2, 1]])` gives `[[], [0, 1]]` instead of `[[0, 1], []]`. Removing a bridge made the "tree" cheaper instead of impossible.
- Drop `i not in critical` from the pseudo-critical test: the critical edges 0 and 1 show up in both lists.
- A triangle with weights 1, 2, 3: `critical_edges(3, [[0, 1, 1], [1, 2, 2], [0, 2, 3]])` is `[[0, 1], []]`. Make all three weights 1 and every edge becomes pseudo-critical: any two of them form an MST.

**At most k edges, or negative weights: Bellman-Ford, in words.** Dijkstra's "popped means final" breaks with negative edges, and it cannot count edges along a path. Bellman-Ford works in rounds instead: each round relaxes *every* edge, reading the distances from the end of the previous round, so after round i every node knows its cheapest path that uses at most i edges. Cheapest Flights Within K Stops (787) is exactly k + 1 rounds.

```text
dist = {src: 0}
repeat k + 1 times                    (V - 1 times for plain shortest paths, O(V * E))
    new = a copy of dist
    for every edge (u, v, w) with u in dist:
        new[v] = min(new.get(v, inf), dist[u] + w)     # read the OLD dist: one more edge per round
    dist = new
```

### Say it in the interview

> "The edges have different costs, so 'fewest edges' isn't 'cheapest' and plain BFS is out. Bellman-Ford would relax every edge V − 1 times, O(V · E). The weights are non-negative, so I'll use Dijkstra: a min-heap of (distance, node). The node I pop with the smallest distance can't be reached more cheaply later, so it's final and I relax its edges once; entries that went out of date are skipped when they surface. That's O((V + E) log V)."

> "We need every point connected at minimum total cost, which is a minimum spanning tree. The cheapest edge leaving the tree built so far is always safe, so Prim's grows the tree with a heap of (edge cost, point): O(n² log n) on the complete graph of n points."

Point at the stale check, the relax condition, and the line where the answer is read. If the weights are only 0 and 1, say "0-1 BFS" and show the two `append` calls.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree | `graphs/find_critical_and_pseudo_critical_edges_in_minimum_spanning_tree.py` | critical: the MST without it is heavier; pseudo: forcing it in first keeps the MST weight |
| Min Cost to Connect All Points | `graphs/min_cost_to_connect_all_points.py` · `practice/simple/46_min_cost_to_connect_all_points.py` | Prim on the complete graph: heap of (distance, point), skip points already in |
| Minimum Cost to Make at Least One Valid Path in a Grid | `graphs/minimum_cost_to_make_at_least_one_valid_path_in_a_grid.py` | 0-1 BFS: following the arrow costs 0 (front), turning it costs 1 (back) |
| Minimum Moves to Move a Box to Their Target Location | `graphs/minimum_moves_to_move_a_box_to_their_target_location.py` | 0-1 BFS over (box, player) states: walking costs 0, a push costs 1 |
| Minimum Weighted Subgraph With the Required Paths | `graphs/minimum_weighted_subgraph_with_the_required_paths.py` | Dijkstra from src1, src2, and from dest on reversed edges; minimise over the meeting node |
| Network Delay Time | `graphs/network_delay_time.py` · `practice/simple/45_network_delay_time.py` | Dijkstra from k; answer = the largest distance, −1 if a node is never reached |
| Swim in Rising Water | `graphs/swim_in_rising_water.py` | bottleneck Dijkstra: key = the highest elevation so far; the first pop of the corner wins |

### Self-check

1. Why is a node's distance final when it is popped (and not stale)? What breaks with a negative edge?
<details><summary>Answer</summary>Every other route to that node has to leave the settled region through some node still in the heap, and that node is already at least as far away. Non-negative edges can only add to that, so no other route can be cheaper. A negative edge can subtract, so a route through a node popped <em>later</em> can still undercut a settled distance.</details>

2. In 0-1 BFS, why must a free move go to the front of the deque?
<details><summary>Answer</summary>The deque must stay sorted by cost, like a heap. A free move has the same cost <code>d</code> as the node being expanded, which is the smallest cost in the deque, so it belongs in front of every <code>d + 1</code> entry. Put at the back, it waits behind more expensive entries, and the target can be popped with a cost that is too high.</details>

3. Prim and Dijkstra look almost identical. What is the one real difference?
<details><summary>Answer</summary>The key in the heap. Dijkstra orders nodes by the whole path cost <code>d + w</code> from the source; Prim orders them by the single edge <code>w</code> that would attach them to the tree.</details>

4. How can one Dijkstra give the distance from <em>every</em> node to <code>dest</code>?
<details><summary>Answer</summary>Reverse every edge. A path x → dest becomes a path dest → x with the same cost, so a single Dijkstra from <code>dest</code> on the reversed graph measures all of them at once.</details>